In [ ]:
# ============================================================
# GENERATES: Particle trajectory data (.txt)
# Output columns: x, y, direction, time
# Model: Spatially heterogeneous-noise Vicsek model
# ============================================================

import os
import numpy as np
from numba import njit
from scipy.spatial import cKDTree


# ============================================================
# NUMBA KERNELS
# ============================================================

@njit(cache=True, fastmath=True)
def average_neighbor_angles(directions, indptr, indices):
    N = directions.shape[0]
    averaged = np.empty(N, dtype=np.float64)

    for i in range(N):
        start, end = indptr[i], indptr[i + 1]
        count = end - start

        if count == 0:
            averaged[i] = directions[i]
            continue

        sx, sy = 0.0, 0.0
        for k in range(start, end):
            theta = directions[indices[k]]
            sx += np.sin(theta)
            sy += np.cos(theta)

        averaged[i] = np.arctan2(sx / count, sy / count)

    return averaged


@njit(cache=True, fastmath=True)
def update_positions(positions, directions, v0, dt, L):
    for i in range(positions.shape[0]):
        positions[i, 0] = (positions[i, 0] + v0 * dt * np.cos(directions[i])) % L
        positions[i, 1] = (positions[i, 1] + v0 * dt * np.sin(directions[i])) % L
    return positions


# ============================================================
# NEIGHBOR LIST
# ============================================================

def build_neighbor_list(tree, positions, radius):
    neighbors = tree.query_ball_point(positions, r=radius)
    lengths = np.fromiter((len(n) for n in neighbors), dtype=np.int64, count=len(neighbors))
    indptr = np.empty(len(neighbors) + 1, dtype=np.int64)
    indptr[0] = 0
    np.cumsum(lengths, dtype=np.int64, out=indptr[1:])
    indices = np.empty(indptr[-1], dtype=np.int64)

    ptr = 0
    for n in neighbors:
        length = len(n)
        if length:
            indices[ptr:ptr + length] = np.asarray(n, dtype=np.int64)
            ptr += length

    return indptr, indices


# ============================================================
# SPATIALLY HETEROGENEOUS NOISE
# ============================================================

def effective_noise(positions, L, eta_inside, srad, eta_outside):
    center = L / 2.0
    dx = positions[:, 0] - center
    dy = positions[:, 1] - center
    inside = dx * dx + dy * dy < srad**2
    return np.where(inside, eta_inside, eta_outside)


# ============================================================
# SINGLE SIMULATION
# ============================================================

def simulate(N, L, eta, R, v0, dt, srad, fac, T, seed=None, save_last_steps=500):
    if seed is not None:
        np.random.seed(seed)

    n_steps = int(T / dt)
    positions = np.random.rand(N, 2) * L
    directions = np.random.rand(N) * 2 * np.pi
    save_start = max(0, n_steps - save_last_steps)
    n_saved = n_steps - save_start

    saved_positions = np.empty((n_saved, N, 2), dtype=np.float64)
    saved_directions = np.empty((n_saved, N), dtype=np.float64)
    saved_times = np.empty(n_saved, dtype=np.float64)

    save_index = 0

    for step in range(1, n_steps):
        tree = cKDTree(positions, boxsize=L)
        indptr, indices = build_neighbor_list(tree, positions, R)
        mean_direction = average_neighbor_angles(directions, indptr, indices)

        eta_eff = effective_noise(positions, L, eta, srad, fac)
        noise = np.random.uniform(-0.5 * eta_eff, 0.5 * eta_eff, N)
        directions = mean_direction + noise
        positions = update_positions(positions, directions, v0, dt, L)

        if step >= save_start:
            saved_positions[save_index] = positions
            saved_directions[save_index] = directions
            saved_times[save_index] = step * dt
            save_index += 1

    return saved_positions, saved_directions, saved_times


# ============================================================
# SAVE TRAJECTORIES
# ============================================================

def save_trajectory(filename, positions, directions, times):
    n_steps, N = positions.shape[:2]
    data = np.empty((n_steps * N, 4), dtype=np.float64)

    for t in range(n_steps):
        start, end = t * N, (t + 1) * N
        data[start:end, 0] = positions[t, :, 0]
        data[start:end, 1] = positions[t, :, 1]
        data[start:end, 2] = directions[t]
        data[start:end, 3] = times[t]

    np.savetxt(filename, data, delimiter="\t", header="x\ty\tdirection\ttime", comments="", fmt=["%.4f", "%.4f", "%.4f", "%.1f"])


# ============================================================
# RUN ALL SIMULATIONS
# ============================================================

def run_simulations():
    N = 1000
    L = 20.0
    eta = 0.0
    R = 1.0
    v0 = 0.5
    dt = 1.0
    T = 500.0
    fac_values = [2]
    srad_values = [8]
    num_repeats = 100
    seed_base = 12345
    save_last_steps = 50
    root_dir = "input_folder"

    os.makedirs(root_dir, exist_ok=True)

    for fac in fac_values:
        fac_dir = os.path.join(root_dir, f"fac{fac:.1f}")
        os.makedirs(fac_dir, exist_ok=True)

        for srad in srad_values:
            srad_dir = os.path.join(fac_dir, f"srad{srad}")
            os.makedirs(srad_dir, exist_ok=True)

            for run in range(1, num_repeats + 1):
                seed = seed_base + run + int(1000 * fac) + 10000 * srad
                positions, directions, times = simulate(N, L, eta, R, v0, dt, srad, fac, T, seed, save_last_steps)
                filename = os.path.join(srad_dir, f"run_{run:02d}.txt")
                save_trajectory(filename, positions, directions, times)

            print(f"Completed fac={fac:.1f}, srad={srad}")

    print(f"Simulation data saved to: {os.path.abspath(root_dir)}")


if __name__ == "__main__":
    run_simulations()